In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128,
                                         shuffle=False)

100%|██████████| 170M/170M [00:13<00:00, 12.4MB/s]


In [3]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

        # Gate scores (same shape as weight)
        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features))

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)  # values in [0,1]
        pruned_weights = self.weight * gates
        return nn.functional.linear(x, pruned_weights, self.bias)

In [4]:
class PrunableNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = PrunableLinear(32*32*3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def get_all_gates(self):
        gates = []
        for module in self.modules():
            if isinstance(module, PrunableLinear):
                gates.append(torch.sigmoid(module.gate_scores))
        return gates

In [5]:
def sparsity_loss(model):
    loss = 0
    for g in model.get_all_gates():
        loss += torch.sum(g)  # L1 norm
    return loss

In [6]:
def train_model(lambda_val, epochs=10):
    model = PrunableNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            cls_loss = criterion(outputs, labels)
            sp_loss = sparsity_loss(model)

            loss = cls_loss + lambda_val * sp_loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"λ={lambda_val} | Epoch {epoch+1} | Loss={total_loss:.2f}")

    return model

In [7]:
def evaluate(model):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, pred = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (pred == labels).sum().item()

    return 100 * correct / total


def calculate_sparsity(model, threshold=1e-2):
    total, pruned = 0, 0

    for g in model.get_all_gates():
        total += g.numel()
        pruned += (g < threshold).sum().item()

    return 100 * pruned / total

In [ ]:
lambdas = [1e-5, 1e-4, 1e-3]
results = []
models = {}

for lam in lambdas:
    model = train_model(lam, epochs=10)

    acc = evaluate(model)
    sparsity = calculate_sparsity(model)

    results.append((lam, acc, sparsity))
    models[lam] = model

λ=1e-05 | Epoch 1 | Loss=3748.11
λ=1e-05 | Epoch 2 | Loss=3197.10
λ=1e-05 | Epoch 3 | Loss=2771.43
λ=1e-05 | Epoch 4 | Loss=2427.53
λ=1e-05 | Epoch 5 | Loss=2153.90
λ=1e-05 | Epoch 6 | Loss=1938.11
λ=1e-05 | Epoch 7 | Loss=1764.74
λ=1e-05 | Epoch 8 | Loss=1624.13


In [ ]:
print("\nLambda | Accuracy | Sparsity (%)")
for r in results:
    print(f"{r[0]} | {r[1]:.2f}% | {r[2]:.2f}%")

In [ ]:
best_lambda = max(results, key=lambda x: x[1])[0]
best_model = models[best_lambda]

all_gates = []
for g in best_model.get_all_gates():
    all_gates.extend(g.detach().cpu().numpy().flatten())

plt.hist(all_gates, bins=50)
plt.title(f"Gate Distribution (λ={best_lambda})")
plt.xlabel("Gate Value")
plt.ylabel("Frequency")
plt.show()